# 02 — Model 1.2 internal chain ablation

**Question.** Which emission noise channels should the per-KC cell chains carry: guess, slip, both, or neither — and at which bounds?

The chains train and evaluate on the **annotated per-KC cell correctness** (every realized cell, the adaptive stream). Question correctness never enters at any point: this is a self-contained ablation of the conceptual layer, upstream of any KC-to-answer bridge.

**File layout:**

* The nine ablation models: `scripts/model_1_2_internal_chain.py`
* The leave-one-participant-out harness: `scripts/evaluator.py` (retains per-fold fitted models in `ev.fold_models`)
* The dataset: `data/data_annotated.csv`
* The loader: `scripts/data.py` (splits the KC list columns and drops the Q0 warm-up)

**The nine arms:**

| Model class | Guess channel | Slip channel |
|---|---:|---:|
| `Model_1_2_Internal_Slip_And_Guess` | ≤ 0.30 | ≤ 0.30 |
| `Model_1_2_Internal_GuessOnly` | ≤ 0.30 | Closed |
| `Model_1_2_Internal_SlipOnly` | Closed | ≤ 0.30 |
| `Model_1_2_Internal_Neither` | Closed | Closed |
| `Model_1_2_Internal_GuessOnly_G_Free` | ≤ 0.85 | Closed |
| `Model_1_2_Internal_SlipOnly_S_Free` | Closed | ≤ 0.85 |
| `Model_1_2_Internal_Slip_And_Guess_G_Free` | ≤ 0.85 | ≤ 0.30 |
| `Model_1_2_Internal_Slip_And_Guess_S_Free` | ≤ 0.30 | ≤ 0.85 |
| `Model_1_2_Internal_Slip_And_Guess_S_And_G_Free` | ≤ 0.85 | ≤ 0.85 |

Closed channels are pinned at the numerical floor (`1e-4`).

**Protocol.** 26-fold leave-one-participant-out, predict-before-update per cell, 768 cell targets. Headline metrics: AUC and log loss (threshold-free); AUPRC with *wrong* as the positive class (minority detection; no-skill floor = wrong prevalence); balanced accuracy; accuracy and F1 for convention.

In [1]:
import pandas as pd
from IPython.display import display

from scripts.data import load_data
from scripts.evaluator import Evaluator
from scripts.model_1_2_internal_chain import (
    Model_1_2_Internal_Slip_And_Guess,
    Model_1_2_Internal_GuessOnly,
    Model_1_2_Internal_SlipOnly,
    Model_1_2_Internal_Neither,
    Model_1_2_Internal_GuessOnly_G_Free,
    Model_1_2_Internal_SlipOnly_S_Free,
    Model_1_2_Internal_Slip_And_Guess_G_Free,
    Model_1_2_Internal_Slip_And_Guess_S_Free,
    Model_1_2_Internal_Slip_And_Guess_S_And_G_Free,
)

DATA = "data/data_annotated.csv"
df = load_data(DATA)

## 1. Run

Run all nine arms through the shared harness. Each arm's per-fold fitted models remain available in its evaluator's `fold_models`.

In [2]:
ARMS = [
    Model_1_2_Internal_Slip_And_Guess,
    Model_1_2_Internal_GuessOnly,
    Model_1_2_Internal_SlipOnly,
    Model_1_2_Internal_Neither,
    Model_1_2_Internal_GuessOnly_G_Free,
    Model_1_2_Internal_SlipOnly_S_Free,
    Model_1_2_Internal_Slip_And_Guess_G_Free,
    Model_1_2_Internal_Slip_And_Guess_S_Free,
    Model_1_2_Internal_Slip_And_Guess_S_And_G_Free,
]

evs, rows = {}, []
for cls in ARMS:
    ev = Evaluator(cls, df, model_kwargs={"n_restarts": 3}).run()
    evs[cls.__name__] = ev
    rows.append({"model": cls.__name__, **{k: round(float(v), 4) for k, v in ev.metrics.items()}})
    print("done", cls.__name__)
results = pd.DataFrame(rows).set_index("model")

done Model_1_2_Internal_Slip_And_Guess
done Model_1_2_Internal_GuessOnly
done Model_1_2_Internal_SlipOnly
done Model_1_2_Internal_Neither
done Model_1_2_Internal_GuessOnly_G_Free
done Model_1_2_Internal_SlipOnly_S_Free
done Model_1_2_Internal_Slip_And_Guess_G_Free
done Model_1_2_Internal_Slip_And_Guess_S_Free
done Model_1_2_Internal_Slip_And_Guess_S_And_G_Free


## 2. Results

### 2.1 Headline metrics across all models

Metrics are calculated from the 768 pooled out-of-fold cell predictions:

* **AUC:** How well the model ranks correct cells above wrong cells across all thresholds; 0.5 represents chance and higher is better.
* **AUPRC-wrong:** Precision–recall performance when `wrong` is treated as the positive class and scored with $1-p(\text{correct})$; higher is better, with the wrong-cell prevalence (0.164) as the no-skill reference.
* **Balanced accuracy:** Mean of the correct-cell recall and wrong-cell recall at the 0.5 threshold; it gives both classes equal weight.
* **Log loss:** Quality and calibration of the predicted probabilities; lower is better, with 0.4463 as the constant base-rate reference.
* **Accuracy:** Proportion of cell labels classified correctly at the 0.5 threshold.
* **F1:** Harmonic mean of precision and recall with `correct` as the positive class.
* **wrong_n:** Number of wrong-cell targets in the pooled evaluation.
* **n:** Total number of evaluated cell targets.

In [3]:
cols = ["auc", "auprc_wrong", "bal_acc", "log_loss", "accuracy", "f1", "wrong_n", "n"]
results[cols].sort_values("auc", ascending=False)

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,wrong_n,n
model,,,,,,,,
Model_1_2_Internal_GuessOnly_G_Free,0.7056,0.2742,0.5000,0.4110,0.8359,0.9106,126.0,768.0
Model_1_2_Internal_GuessOnly,0.7033,0.2761,0.5816,0.4766,0.7643,0.8583,126.0,768.0
Model_1_2_Internal_Slip_And_Guess_G_Free,0.7013,0.2684,0.4992,0.4157,0.8346,0.9099,126.0,768.0
Model_1_2_Internal_Slip_And_Guess_S_And_G_Free,0.7013,0.2684,0.4992,0.4157,0.8346,0.9099,126.0,768.0
Model_1_2_Internal_Neither,0.6857,0.2908,0.5607,0.9985,0.8255,0.9015,126.0,768.0
Model_1_2_Internal_Slip_And_Guess,0.6575,0.2847,0.5225,0.4296,0.8255,0.9032,126.0,768.0
Model_1_2_Internal_Slip_And_Guess_S_Free,0.6575,0.2847,0.5225,0.4296,0.8255,0.9032,126.0,768.0
Model_1_2_Internal_SlipOnly,0.5270,0.2020,0.4969,0.4426,0.8307,0.9075,126.0,768.0
Model_1_2_Internal_SlipOnly_S_Free,0.5270,0.2020,0.4969,0.4426,0.8307,0.9075,126.0,768.0


### 2.2 Fitted parameters across all models

The table reports the mean fitted $L_0$, $T$, guess ($g$), and slip ($s$) for every model–KC combination. Each value averages the 26 separately fitted leave-one-participant-out models, so the display contains 45 rows (nine models × five KCs). $L_0$ is initial mastery, $T$ is the learning transition probability, $g$ is the probability of a correct cell while unmastered, and $s$ is the probability of a wrong cell while mastered. Closed guess or slip channels remain at the numerical floor of 0.0001. These are fold means, not confidence intervals or parameter ranges.

In [4]:
# Mean fitted parameters across the 26 folds for every model and KC.
parameter_records = []
for model_name, ev in evs.items():
    for held_out_participant, model in ev.fold_models.items():
        for kc, chain in model.chains.items():
            parameter_records.append(
                {
                    "model": model_name,
                    "held_out_participant": held_out_participant,
                    "kc": kc,
                    "L0": chain.L0,
                    "T": chain.T,
                    "g": chain.g,
                    "s": chain.s,
                }
            )

fold_parameters = pd.DataFrame(parameter_records)
parameter_means = (
    fold_parameters.groupby(["model", "kc"])[["L0", "T", "g", "s"]]
    .mean()
    .round(4)
)
parameter_means

L0  \
model                                          kc                              
Model_1_2_Internal_GuessOnly                   kc1_sample_space       0.7496   
                                               kc2_conditioning       0.4959   
                                               kc3_joint_chain        0.2919   
                                               kc4_total_probability  0.5330   
                                               kc5_bayes_update       0.4511   
Model_1_2_Internal_GuessOnly_G_Free            kc1_sample_space       0.5944   
                                               kc2_conditioning       0.3695   
                                               kc3_joint_chain        0.1672   
                                               kc4_total_probability  0.5134   
                                               kc5_bayes_update       0.4364   
Model_1_2_Internal_Neither                     kc1_sample_space       0.8824   
                                               kc2_conditioning       0.7640   
                                               kc3_joint_chain        0.5878   
                                               kc4_total_probability  0.7659   
                                               kc5_bayes_update       0.6865   
Model_1_2_Internal_SlipOnly                    kc1_sample_space       0.9276   
                                               kc2_conditioning       0.9960   
                                               kc3_joint_chain        0.8540   
                                               kc4_total_probability  0.9610   
                                               kc5_bayes_update       0.8878   
Model_1_2_Internal_SlipOnly_S_Free             kc1_sample_space       0.9276   
                                               kc2_conditioning       0.9960   
                                               kc3_joint_chain        0.8540   
                                               kc4_total_probability  0.9610   
                                               kc5_bayes_update       0.8878   
Model_1_2_Internal_Slip_And_Guess              kc1_sample_space       0.8253   
                                               kc2_conditioning       0.9071   
                                               kc3_joint_chain        0.5611   
                                               kc4_total_probability  0.7223   
                                               kc5_bayes_update       0.7266   
Model_1_2_Internal_Slip_And_Guess_G_Free       kc1_sample_space       0.7732   
                                               kc2_conditioning       0.5302   
                                               kc3_joint_chain        0.1660   
                                               kc4_total_probability  0.5754   
                                               kc5_bayes_update       0.4708   
Model_1_2_Internal_Slip_And_Guess_S_And_G_Free kc1_sample_space       0.7732   
                                               kc2_conditioning       0.5302   
                                               kc3_joint_chain        0.1660   
                                               kc4_total_probability  0.5754   
                                               kc5_bayes_update       0.4708   
Model_1_2_Internal_Slip_And_Guess_S_Free       kc1_sample_space       0.8253   
                                               kc2_conditioning       0.9071   
                                               kc3_joint_chain        0.5611   
                                               kc4_total_probability  0.7223   
                                               kc5_bayes_update       0.7266   

                                                                           T  \
model                                          kc                              
Model_1_2_Internal_GuessOnly                   kc1_sample_space       0.3826   
                                               kc2_conditioning       0.1

## 3. Conclusion

* **Only the guess bound ever binds.** The slip bound never constrains any arm: `Model_1_2_Internal_SlipOnly_S_Free` fits identically to `Model_1_2_Internal_SlipOnly`, `Model_1_2_Internal_Slip_And_Guess_S_Free` fits identically to `Model_1_2_Internal_Slip_And_Guess`, and `Model_1_2_Internal_Slip_And_Guess_S_And_G_Free` fits identically to `Model_1_2_Internal_Slip_And_Guess_G_Free`. Once guess is free, fitted slip collapses to approximately 0.00–0.03. Nine arms produce six distinct fits; cell-level noise is guess-shaped rather than slip-shaped.
* **Guess-side arms win ranking.** `Model_1_2_Internal_GuessOnly_G_Free` has the best AUC (approximately 0.706) and the only clamped-or-freed log loss that beats the base-rate reference (approximately 0.411 versus 0.446). Clamped `Model_1_2_Internal_GuessOnly` nearly matches its AUC (approximately 0.703) but has worse calibration (log loss approximately 0.477). `Model_1_2_Internal_SlipOnly` is near chance (AUC approximately 0.527).
* **The freed-arm balanced accuracy of 0.500 is a threshold artifact.** With guess fitted around 0.57–0.76, no prediction falls below 0.5. Threshold-based metrics therefore become uninformative even though threshold-free metrics improve.
* **Minority detection is weak in every arm.** The best AUPRC-wrong is approximately 0.29 against the 0.164 no-skill reference. Cell failures are only mildly predictable from correctness history alone.
* **Adopted configuration: `Model_1_2_Internal_Slip_And_Guess`.** This is the conventional BKT emission with 0.3/0.3 clamps. Its costs remain visible: lower cell-level AUC and a clamp-induced kc2 entry-prior pathology. The freed-guess result is retained as a finding rather than silently discarded.

In [5]:
# Per-KC results for the adopted arm.
from scripts.evaluator import _metrics

predictions = evs["Model_1_2_Internal_Slip_And_Guess"].predictions
per_kc_results = pd.DataFrame(
    [
        {"kc": kc, **{k: round(float(v), 3) for k, v in _metrics(group.y_true, group.p_pred).items()}}
        for kc, group in predictions.groupby("kc")
    ]
).set_index("kc")
per_kc_results[["auc", "auprc_wrong", "log_loss", "wrong_n", "n"]]

,auc,auprc_wrong,log_loss,wrong_n,n
kc,,,,,
kc1_sample_space,0.679,0.153,0.240,9.0,152.0
kc2_conditioning,0.349,0.121,0.437,23.0,156.0
kc3_joint_chain,0.581,0.269,0.519,36.0,165.0
kc4_total_probability,0.660,0.355,0.445,24.0,132.0
kc5_bayes_update,0.632,0.318,0.496,34.0,163.0


## 4. Save selected internal chain model

In [7]:
import os
import json
from scripts.model_1_2_internal_chain import save_internal_chain_from_evaluator

out_dir = "cache/model_1_2_internal_chain"
save_internal_chain_from_evaluator(evs['Model_1_2_Internal_Slip_And_Guess'], out_dir)

'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'